# Team 02 — Feature set (Deliverable 3)

Builds the team's one feature table from the raw AA files, following everyone's D3 hand-offs, and saves it to
`data/processed/team02_features.csv` (git-ignored — never commit it). Every D4 model reads that file.

| Step | Decision | From |
|---|---|---|
| Drop `OD_REV_USD`, `SF_LOAD_TMS`, `Unnamed: 0` | leak / load timestamp / row index | Sahith (leakage) |
| Target `OD_PAX`, keep large values | busy routes are real | Aidan (target) |
| One row per route per month; missing months are left missing, not filled with zero | open PM question | Chau Ha (missingness), Akshith |
| Date and lag features | only use earlier months | Akshith (time axis) |
| Country + international flag from the airport table | encoding plan | categoricals deep-dive |
| Categoricals stay as text here | each model encodes them inside its own pipeline, fit on training months only | team rule |

Built by the lead (Akshith Reddy Sheri), taking over this piece from Aditi Walvekar.
**NDA:** this notebook prints summaries only. Clear outputs before committing.

## 1. Load

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 40)
plt.rcParams['figure.figsize'] = (10, 4)

TARGET = 'OD_PAX'
RANDOM_STATE = 42
TEST_MONTHS = 12          # same hold-out as the team validation scheme: last 12 departure months

# Find the repo root (works whether Jupyter starts in the repo or in this folder)
repo_root = next((f for f in [Path.cwd(), *Path.cwd().parents]
                  if (f / 'DATA.md').is_file() and (f / 'data').is_dir()), None)
if repo_root is None:
    raise FileNotFoundError('Open this notebook from inside the project repository.')

def find_file(name):
    for folder in [repo_root / 'data' / 'raw', repo_root / 'data']:
        if (folder / name).is_file():
            return folder / name
    raise FileNotFoundError(f'Put {name} in data/raw/ (git-ignored).')

pax = pd.read_csv(find_file('AA_PAX_FORECASTING.csv'))
try:
    airports = pd.read_csv(find_file('AA_AIRPORT_INFO.csv'))
except FileNotFoundError:
    airports = None
    print('Airport reference file not found -- country columns will be UNKNOWN.')

required = {'DEP_YEAR', 'DEP_MONTH', 'ORG_AIRPRT', 'DES_AIRPRT', TARGET}
missing_cols = required - set(pax.columns)
if missing_cols:
    raise ValueError(f'Passenger file is missing expected columns: {sorted(missing_cols)}')
print('Files loaded locally. No individual records are displayed in this notebook.')

## 2. Drop leak / ID columns, one row per route-month, attach airport country

In [ ]:
# Agreed in D3 hand-offs (Sahith): these are a leak, a load timestamp and a row index -> drop.
DROP_COLS = ['OD_REV_USD', 'SF_LOAD_TMS', 'Unnamed: 0']
pax = pax.drop(columns=[c for c in DROP_COLS if c in pax.columns])

pax[TARGET] = pd.to_numeric(pax[TARGET], errors='coerce')
pax['period'] = pd.to_datetime(dict(year=pax['DEP_YEAR'], month=pax['DEP_MONTH'], day=1), errors='coerce')
bad = pax['period'].isna() | pax[TARGET].isna()
print(f'rows dropped for an unreadable date or target: {bad.mean():.2%}')

# One row per route per month (duplicates, if any, are summed)
rm = pax[~bad].groupby(['ORG_AIRPRT', 'DES_AIRPRT', 'period'], as_index=False)[TARGET].sum()
rm['ORG_AIRPRT'] = rm['ORG_AIRPRT'].astype(str)
rm['DES_AIRPRT'] = rm['DES_AIRPRT'].astype(str)
rm['route'] = rm['ORG_AIRPRT'] + '-' + rm['DES_AIRPRT']
rm['DEP_YEAR'], rm['DEP_MONTH'] = rm['period'].dt.year, rm['period'].dt.month

# Country of each airport, from the airport reference table
if airports is not None and {'AIRPRT_CD', 'CNTRY_CD'} <= set(airports.columns):
    country = (airports.assign(AIRPRT_CD=airports['AIRPRT_CD'].astype(str))
               .drop_duplicates('AIRPRT_CD').set_index('AIRPRT_CD')['CNTRY_CD'].astype(str))
else:
    country = pd.Series(dtype=str)
for side in ['ORG', 'DES']:
    rm[f'{side}_CNTRY_CD'] = rm[f'{side}_AIRPRT'].map(country).fillna('UNKNOWN')
known = (rm['ORG_CNTRY_CD'] != 'UNKNOWN') & (rm['DES_CNTRY_CD'] != 'UNKNOWN')
rm['is_international'] = (rm['ORG_CNTRY_CD'] != rm['DES_CNTRY_CD']).astype(float).where(known)
print(f'route-months with an airport missing from the reference table: {(~known).mean():.2%}')

## 3. Date and lag features

All lag features look **backwards only**: "same month last year" for a row in month *t* is the same route's
passengers in month *t − 12*. The two-years-ago versions are there for the D5 two-year-ahead forecast, where the
last-year value will not exist yet for the second year. A missing lag means "no row for that route-month" — it is
left as missing, not zero (open PM question).

In [ ]:
start = rm['period'].min()
rm['month_sin'] = np.sin(2 * np.pi * rm['DEP_MONTH'] / 12)
rm['month_cos'] = np.cos(2 * np.pi * rm['DEP_MONTH'] / 12)
rm['months_since_start'] = (rm['period'].dt.year - start.year) * 12 + rm['period'].dt.month - start.month

def add_lag(frame, years, name):
    past = frame[['ORG_AIRPRT', 'DES_AIRPRT', 'period', TARGET]].copy()
    past['period'] = past['period'] + pd.DateOffset(years=years)   # move the past forward onto the row it informs
    return frame.merge(past.rename(columns={TARGET: name}), on=['ORG_AIRPRT', 'DES_AIRPRT', 'period'], how='left')

rm = add_lag(rm, 1, 'pax_same_month_last_year')
rm = add_lag(rm, 2, 'pax_same_month_2_years_ago')
rm['route_seen_last_year'] = rm['pax_same_month_last_year'].notna().astype(int)
rm['route_seen_2_years_ago'] = rm['pax_same_month_2_years_ago'].notna().astype(int)
print(f'rows with a last-year value: {rm["route_seen_last_year"].mean():.1%} | '
      f'with a two-years-ago value: {rm["route_seen_2_years_ago"].mean():.1%} '
      '(the first one / two years can never have them)')

## 4. Final columns and safety checks

In [ ]:
ID_COLS = ['period', 'route']                                   # for splitting and grouping, not model inputs
CAT_FEATURES = ['ORG_AIRPRT', 'DES_AIRPRT', 'ORG_CNTRY_CD', 'DES_CNTRY_CD']
NUM_FEATURES = ['DEP_YEAR', 'DEP_MONTH', 'month_sin', 'month_cos', 'months_since_start', 'is_international',
                'pax_same_month_last_year', 'route_seen_last_year',
                'pax_same_month_2_years_ago', 'route_seen_2_years_ago']
features = rm[ID_COLS + CAT_FEATURES + NUM_FEATURES + [TARGET]].sort_values(['period', 'route']).reset_index(drop=True)

# Checks -- each one stops the notebook if something is wrong
assert not set(DROP_COLS) & set(features.columns), 'a dropped leak / ID column came back'
assert TARGET not in CAT_FEATURES + NUM_FEATURES, 'the target must not be a feature'
assert not features.duplicated(['route', 'period']).any(), 'more than one row per route-month'
assert features[CAT_FEATURES].notna().all().all(), 'categoricals should never be missing'
# A lag must equal the target of the same route exactly 12 / 24 months earlier (spot check on a sample)
lookup = features.set_index(['route', 'period'])[TARGET]
for col, yrs in [('pax_same_month_last_year', 1), ('pax_same_month_2_years_ago', 2)]:
    s = features.dropna(subset=[col]).sample(min(2000, features[col].notna().sum()), random_state=RANDOM_STATE)
    past = lookup.reindex(list(zip(s['route'], s['period'] - pd.DateOffset(years=yrs)))).to_numpy()
    assert np.allclose(past, s[col].to_numpy()), f'{col} does not match the earlier month'
print('all checks passed')

register = pd.DataFrame({
    'type': ['id'] * len(ID_COLS) + ['categorical'] * len(CAT_FEATURES) + ['numeric'] * len(NUM_FEATURES) + ['target'],
    'missing %': features[ID_COLS + CAT_FEATURES + NUM_FEATURES + [TARGET]].isna().mean().mul(100).round(1).values,
}, index=ID_COLS + CAT_FEATURES + NUM_FEATURES + [TARGET])
display(register)

## 5. Save (local only, git-ignored)

D4 notebooks load it with:

```python
features = pd.read_csv(repo_root / 'data' / 'processed' / 'team02_features.csv', parse_dates=['period'])
```

and split on `period` (last 12 months = test). Encoders (one-hot, target encoding, ordinal) go **inside** each
model's pipeline and are fit on the training months only.

In [ ]:
out_dir = repo_root / 'data' / 'processed'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'team02_features.csv'
features.to_csv(out_path, index=False)
print(f'saved {out_path.relative_to(repo_root)} -- {features.shape[1]} columns. This file is git-ignored; never commit it.')

### Before committing
Kernel → Restart & Clear Output, then save. `git status` must show only this notebook — never anything under `data/`.